# Finetuning

In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.0 MB/s eta 0:00:00


Prompting

In [18]:
import os
from groq import Groq

In [19]:
# Set your Groq API Key
groq_api_key = "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"
os.environ["GROQ_API_KEY"] = groq_api_key

# Create client
client = Groq(api_key=groq_api_key)

def prompt_based_query(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # or another Groq-supported model
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_query}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# Example
answer = prompt_based_query("What is Machine Learning?")
print(answer)

**Machine Learning (ML)** is a subset of Artificial Intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to automatically improve their performance on a task by learning from experience, rather than relying on human intervention.

**Key Characteristics of Machine Learning:**

1. **Data-Driven**: ML relies on large amounts of data to learn patterns, relationships, and trends.
2. **Algorithmic**: ML uses algorithms to analyze data and make predictions or decisions.
3. **Self-Improving**: ML models can improve their performance over time as they receive more data and learn from their mistakes.
4. **Autonomous**: ML models can operate independently, making decisions without human intervention.

**Types of Machine Learning:**

1. **Supervised Learning**: The model is trained on labeled data to learn the relationship between input and output.
2. **Unsupervised Learning**: The model i

## RAG

Load Document

In [20]:
!pip install langchian faiss-cpu openai langchain_community tiktoken

In [22]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/photosynthesis.txt')
documents=loader.load()

In [23]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(documents, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
#step 3 Retrieval + Generation
def rag_query(query):
  docs=vector_db.similarity_search(query,k=3)
  context=" ".join(doc.page_content for doc in docs)
  response=client.chat.completions.create(
       model="llama-3.3-70b-versatile",
       messages=[
           {"role":"system","content":"use the provided context to answer"},
           {"role":"user","content": f"context:{context} \n\nQuestions:{query}"}
       ]
       )
  return response.choices[0].message.content

In [25]:
print(rag_query("Why is photosynthesis important for life on Earth?"))

Photosynthesis is important for life on Earth because it:

1. **Produces oxygen**: Photosynthesis releases oxygen into the atmosphere, which is essential for the survival of most living organisms.
2. **Supports food chains**: Photosynthesis is the primary source of energy for nearly every food web on the planet, as autotrophs (organisms that produce their own food through photosynthesis) are the foundation of most food chains.
3. **Sustains biodiversity**: The energy produced by photosynthesis supports the diversity of life on Earth, from plants and animals to microorganisms.
4. **Provides fossil fuels**: Photosynthesis is responsible for the formation of fossil fuels, such as coal, oil, and natural gas, which power industrial society.
5. **Regulates the atmosphere**: Photosynthesis helps to remove carbon dioxide from the atmosphere and release oxygen, which helps to regulate the Earth's climate.

Without photosynthesis, life on Earth would not be possible as we know it. Most organisms

# Fine Tuning(PEFT +LoRA)

In [10]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [26]:
#step 1 : Load model
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
import bitsandbytes as bnb
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)
#define the bitsandbytes config for 8 bit quantization
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [28]:
#step 2 : Apply LoRA(PEFT)
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
#preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [29]:
#step 3: dataset preperation
from datasets import Dataset
data = [
    {"text":"Q: What is photosynthesis?\nA: Photosynthesis is the process by which plants use sunlight to produce food and oxygen."},
    {"text":"Q: Why is photosynthesis important?\nA: Photosynthesis provides food and oxygen necessary for sustaining life on Earth."}
]
dataset=Dataset.from_list(data)

In [30]:
#step 4: tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True,padding="max_length")
tokenized_dataset=dataset.map(tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [31]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=2, training_loss=5.350971221923828, metrics={'train_runtime': 17.7714, 'train_samples_per_second': 0.225, 'train_steps_per_second': 0.113, 'total_flos': 50903717511168.0, 'train_loss': 5.350971221923828, 'epoch': 2.0})

In [32]:
#step 6: inferences
def generate_response(prompt):
  inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
  outputs=model.generate(**inputs,max_new_tokens=100)
  return tokenizer.decode(outputs[0])

In [34]:
print(generate_response("Why is photosynthesis important for life on Earth?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s> Why is photosynthesis important for life on Earth?</s>
